# Lesson 1A 

In this class we will learn how to work with networks in R:

- Creating networks: manually and from files
- Saving networks: formats 

## Introduction to graphs in R

In this course we will work with networks using a small set of key R libraries. 

We will use **igraph** as the main tool to create and analyse networks, since it allows us to work with graphs, compute network metrics, and apply different algorithms. 

Data manipulation will be done using the **tidyverse**, which makes it easier to handle tables associated with networks. 

For visualisation, we will use **ggraph**, which allows flexible and clear network representations.
 
In addition, for ecological networks such as plant–pollinator systems, we will also use specialised packages like **bipartite**, designed specifically for the analysis of these types of networks.


### Importing the required libraries


In [ ]:
# If needed (run once):
# install.packages(c("igraph","tidyverse","tidygraph","ggraph"))

library(igraph)
library(tidyverse)
library(tidygraph)
library(ggraph)
library(bipartite)

source("Functions.R") #include some helpers


## Creating simple graphs
**igraph** offers many ways to create a graph. The simplest one is the function `make_empty_graph()`:

In [ ]:
g <- make_empty_graph()

To add one or more vertices to an existing graph, use `add_vertices()`

In [ ]:
g <- add_vertices(g, 37) 

#Note: If you try to add edges to vertices with invalid IDs (i.e., you try to add an edge to vertex 38 when the graph has only 37 vertices), 
#igraph shows an error:

Similarly, to add vertices you can use `add_edges()`. Edges are added by specifying the source and target vertex IDs for each edge. This call added three edges, one connecting vertices 1 and 35, one connecting vertices 1 and 36, and one connecting vertices 34 and 37.

In [ ]:
g <- add_edges(g, edges = c(1,35, 1,36, 34,37))

The most common way to create a graph is make_graph(), which constructs a network based on specified edges. For example, to make an undirected graph with 5 nodes (numbered 1 to 5) and 4 edges:

In [ ]:
# from list of edges (it assumes n=maximum value of elements in )
EdgeList1 <- c(1,2, 1,3, 2,3, 3,5)
g1 <- make_graph(edges=EdgeList1, directed=FALSE)
g1$name <- "Example network" #we name the network

We can print the graph to get a summary of its nodes and edges:

In [ ]:
g1

This means: Undirected graph with 5 vertices and 4 edges, with the exact edges listed out. If the graph has a [name] attribute, it is printed as well.

We can retrieve the nodes(vertices) as `V(g)` and the links(edges) as `E(g)`, to iterate trouhg them.

In [ ]:
print(V(g1))
print(E(g1))

And the general information with summary

In [ ]:
print(class(g1))
summary(g1)

We can visualize the network with `plot(g)`

In [ ]:
plot(g1)

There are many optional arguments to the draw function to customize the appearance.

In [ ]:
plot(g1, vertex.color="blue", vertex.size=10, 

     vertex.frame.color="gray", vertex.label.color="black", 

     vertex.label.cex=0.8, vertex.label.dist=2, edge.curved=0.2) 


<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 1</h3>
Create the undirected graph corresponding to this simple network

![title](./images/figure1.png)
</div>

In [ ]:
# write your code here and run it ################
#g2<-

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex1.R")


We now have an undirected graph with 5 vertices and 4 edges. 
Vertex and edge IDs are **always** contiguous, so if you delete a vertex all subsequent vertices will be renumbered. 
When a vertex is renumbered, edges are not renumbered, but their source and target vertices will be.
Let's use `delete_vertices()` and `delete_edges()` to perform these operations.

In [ ]:
g2

In [ ]:
edge_id_to_delete <- get_edge_ids(g2, c(2, 3))
edge_id_to_delete

In [ ]:
g2 <- delete_edges(g2, edge_id_to_delete)
g2

In [ ]:
plot(g2)

## Creating non-simple Graphs - Edge properties
As we have seen, we can add different attributes to edges and nodes to enrich the information contained in the network. 
Usually much information can be incuding regarding the "interaction" between a couple of nodes. 
The most frequent examples are direction and weigth.


### Directed graphs
Unless otherwise specified, we assume graph edges are undirected -- they are symmetric and go both ways. 
But some relationships, e.g. predator-prey relationships, are asymmetric and best represented as directed graphs. 
In igraph we can make a directed graph by indicating in the construction that edges are NOT simmetric. 
Let's create a **Directed graph**

In [ ]:
edges <- c(
  1, 2,
  1, 3,
  2, 3,
  2, 4,
  3, 4,
  3, 5,
  4, 5,
  4, 6,
  5, 6,
  6, 3
)

# Create the directed graph
dg <- make_graph(edges = edges, directed = TRUE)

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>
<h3> Exercise 2</h3>
Create the directed graph corresponding to this simple network and plot it

![title](./images/figure2.jpg)
</div>

In [ ]:
# write your code here and run it ################
#g3<-

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex2.R")


### Weighted graphs
We can add further information to the edges between nodes. Including the weigth of the interaction is very frequent. We can create a weighted network by adding the `ẁeigth` attribute to the edges in the network 


In [ ]:
E(g3)$weight <- c(2, 4, 3, 2, 1)
plot(g3,vertex.color="white",vertex.frame.color="black", vertex.label.color="black",edge.color="black",edge.width = E(g3)$weight,vertex.size=20) 

## Creating non-simple Graphs - Node properties
In some cases we have different clases of nodes, for example plants and pollinators, or seed and dispersers. In this case the interactions only take place between nodes of different sets. These types of networks are called **bipartite networks** (because we have two different types of nodes), or multipartite networks if we have more than two different sets. Now, the attribute belongs to the **node**, not to the link. 

### Bipartite graphs
This type of networks are very frequent in ecology. While *igraph* does not have a dedicated bipartite graph class, the normal graph classes can be used to represent bipartite graphs. However, you have to keep track of which set each node belongs to, and make sure that there is no edge between nodes of the same set. The convention used in *igraph* is to use a node attribute named `type` with values TRUE or FALSE to identify the sets each node belongs to. This convention is not enforced in the code of *igraph* functions, it’s only a recommendation. Let's see how we can create a bipartite network using *igraph*.

In [ ]:
B <- make_empty_graph(directed=FALSE) + vertices(c("P1","P2","P3","P4", "A1","A2","A3","A4","A5")) #two plants and three animals
V(B)$type <- c(TRUE, TRUE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE) #plants have type TRUE (bottom set) and animals type FALSE (upper set). We NEED to specify it
B <- add_edges(B,c("P1","A1",  "P1","A2", "P1","A3", "P2","A2", "P2","A3",  "P2","A5", "P3","A2", "P3","A4", "P3","A5", "P4","A5"))
is_bipartite(B)

In [ ]:
plot(B, layout = layout_as_bipartite)

#### Obtaining the bipartite sets
To retrieve the two sets, we just have to filter nodes with type==FALSE (top set), and type==TRUE (bottom set)

In [ ]:
top_nodes <- V(B)$name[V(B)$type == FALSE] #animals
bottom_nodes <- V(B)$name[V(B)$type == TRUE] #plants

In [ ]:
top_nodes

#### Bipartite projections
One of the most frequent uses of weighted networks is obtaining the weighted one-mode projections of bipartite networks. 

*igraph* has the function `bipartite_projection(g)`to obtain the one-mode projection of bipartite networks. 
These unipartite networks tell us how many shared partners each animal/plant has with the other rest of species in its same guild. 
Let's see how it works

In [ ]:
proj <- bipartite_projection(B)
B_animals <- proj$proj1 #type==FALSE
B_plants  <- proj$proj2 #type==TRUE

In [ ]:
plot(B_animals, edge.width = E(B_animals)$weight^2,edge.label = E(B_animals)$weight)

Now let's try one with node attributes too, like a bipartite network

## Network representations

We have seen only the *graph* representation of the networks, but that is not the only way in which we can represent the interaction network. There are three ways in which interactions can be stored:
- graph  
- matrix  
- interaction list

### Networks as Dataframes - Matrix representation

The information that we have stored in the graph can, most times, be also stored in a dataframe. In these dataframes each row and each column represent one species, and the cell value represents the value of the intearction between them. Let's see how we can move from one format to another with some examples

#### Adjacency matrix

The adjacency matrix is a square matrix of dimension NxN (it has as many rows and columns as species), and contains the information of all the possible pairwise interaction among all the species in the community. Each element $A_{ij}$ represents the effect of $i$ over $j$, and are equal to $0$ when there is no interaction between $i$ and $j$, and different from 0 when there is an interaction. For those elements that are not zero, if the network is **unweighted** they will be all equal to 1, and if the networks is **weighted** they will be a number representing the intensity or frequency of the interaction. 

Let's see how we can obtain the adjacency matrices from unipartite graphs (where all nodes are of the same tyoe and can be connected among them).

In [ ]:
#unweighted example
A1 <- as.matrix(as_adjacency_matrix(g3))
A1
rowSums(A1)
colSums(A1)

In [ ]:
#weighted example
A2 <- as.matrix(as_adjacency_matrix(g3, attr = "weight")) #the attribute weight must be defined before and contain the information
A2


In [ ]:
visweb(A2,labsize=0.2)
plot_foodweb_matrix(A2) #custom fucntion to plot the matrix, because bipartite ONLY works in BIPARTITE networks

Now let's see the adjacency matrix of a plant-animal mutualistic network. Remember that the adjacency includes ALL pairwise interactions

In [ ]:
A3 <- as.matrix(as_adjacency_matrix(B))
A3

#### Incidence matrix

As you have jsut seen the natural way to encode the matrix information in bipartite networks is to just focus on part of the adjacency matrix, a subselection with dimension $N_{P}xN_{A}$. This matrix is called the incidence matrix, where plants appear in rows and animals in columns. We can generate the incidence matrix with the function `as_incidence_matrix(g)`

In [ ]:
#unweighted networks
M1 <- t(as_biadjacency_matrix(B))
M1
rowSums(M1)
colSums(M1)

In [ ]:
#if we had frequencies as weight
#M <- t(as_incidence_matrix(B),attr = "weight")

The library *bipartite*, that we are going to use to work with bipartite networks, is based in this matrix representation: M1. Most functions in that library use the Incidence matrix as variable. For example, let's plot this network using the functions from bipartite. 
For more information on how to plot with bipartite see https://cran.r-project.org/web/packages/bipartite/vignettes/PlottingWithBipartite.html

In [ ]:
plotweb(M1,spacing = "auto", text.rot = 90, lab_size = 4) #srt = 90, text_size=0.75

In [ ]:
visweb(M1)

We have seen how we can go from Graph to Dataframe, and its representations, but we can also go from Dataframe to Graph format. 
In fact, this is very usefull to read networks from files as the adjacency and incidence matrices are the most frequent way in which interaction networks are shared. 

Let's see how we can go from a Dataframe to a Graph format

### Graph from Adjacency matrices

Remember the adjacency matrix of the simple foodweb we saw before. We can construct a graph representation like this:

In [ ]:
A1

In [ ]:
g1 <- graph_from_adjacency_matrix(
  A1,
  mode = "directed",   # or "undirected"
  weighted = FALSE,     # FALSE if binary
  diag = FALSE         # ignore self-loops
)
plot(g1)

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 3</h3>
Create the directed graph corresponding to adjacency matrix A2, and plot it


In [ ]:
A2

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex3.R")

### Graph from Incidence matrices

Remeber the matrix representation of the incidence matrix of the plant-animal mutualistic network we saw. We can create the graph representation as follows

In [ ]:
M1

In [ ]:
B1 <- graph_from_biadjacency_matrix(t(M1)) #we use the traspose because FALSE nodes appear as rows in the amtrix, but as upper level nodes in the bipartite graph.
is_bipartite(B1)

In [ ]:
plot(B1, layout = layout_as_bipartite)

## Reading networks from adjaceny and incidence matrix files

As most networks are shared in these formats, the most comon way to import networks into your code will be by loading the csv as a dataframe and the converting that dataframe in a graph.
Let's see how we can import some real networks in here.

### Unipartite networks from adjacency matrices

Networks with only one type of nodes can be read from the $NxN$ Adjacency matrix ($A$) like this: 

*Note that you need a csv where the first row and column contains the names of the species. You should allways check that the structure matches what you expect*

In [ ]:
# Read the foodweb from the Mondego Estuary 
FW_Afilename<-"./Data/Mondego_adjacency.csv"
df <- read.csv(FW_Afilename, row.names = 1)
A <- as.matrix(df)
FW <- graph_from_adjacency_matrix(
  A,
  mode = "directed",   # or "undirected"
  weighted = TRUE,     # FALSE if binary
)
plot_as_flux(FW)

### Bipartite networks from Incidence matrices

Bipartite networks of mutualism can be read from the $N_p x N_A$ Incidence matrix ($I$) like this. 

*Again, note that the first row are the animal names and the first column the plant names.You should allways check that the structure matches what you expect*

In [ ]:
# Read the polination network from
B_Ifilename <- "./Data/Herrera_Donana_incidence.csv"
df <- read.csv(B_Ifilename, row.names = 1)
I <- as.matrix(df)
B <- graph_from_biadjacency_matrix(t(I))
is_bipartite(B)
plot(B, layout = layout_as_bipartite)

## Networks as Dataframes - Interaction list

While the matrix form is the most common way to store information, is not the most efficient. When networks are large the matrix form stores a lot of 0s that are not relly needed. There is another format in which interactions can be stored, which is much more efficient for large networks: interaction lists. 

Interaction lists only contain the itneractions that exists in the network, and can also be stored in a dataframe where the first column represent the source node (from), and the second column the sink node (to), with an optional third column that contains information about the link, like for example the weigth or the sign of the interaction. 


### Unipartite networks

Let's see how the itneraction list of the Mondego foodweb looks like

In [ ]:
FW_Ifilename <-"./Data/Mondego_ilist.csv"
FW_Ilist <- read.csv(FW_Ifilename)
head(FW_Ilist)

One can create a directed graph from an interaction list with `

In [ ]:
FW2 <- graph_from_data_frame(FW_Ilist,directed = TRUE)
plot_as_flux(FW2)
#E(FW2)$weight

### Bipartite networks

It is not common to find mutualistic networks in this format, but it can also be used. The important point here is that not all nodes appear in the two columns, rather plants will be in column one, and animals in column two, 

In [ ]:
B_Ifilename<-"./Data/Herrera_Donana_ilist.csv"
B_Ilist <- read.csv(B_Ifilename)
B2 <- graph_from_data_frame(B_Ilist, directed = FALSE)
#however igraph does not directly understands this is a bipartite graph, so we need to specify the node sets
V(B2)$type <- V(B2)$name %in% B_Ilist$plant_sp #Planst as type==TRUE
is_bipartite(B2)

In [ ]:
plot(B2, layout = layout_as_bipartite)

## Save to file
Once you are done creating your network, probably you will want to save it. 

The most useful format for networks with attributes is **GML**, as it allows to keep these attributes, but is readeable. 
Let's see how it works, and how the information is stored. 

You can save (and load) netowkrs using this format like this:

Unipartite networks:

In [ ]:
filename<-"Output/FW.gml"
write_graph(FW, filename, format = "gml")

In [ ]:
#let's read to see how it looks like
FW3 <- read_graph(filename, format = "gml")

And Bipartite netowkrs too: (and they conserve the bipartite information)

In [ ]:
filename<-"Output/B.gml"
write_graph(B, filename, format = "gml")
B3 <- read_graph(filename, format = "gml")
is_bipartite(B3)

It is possible also to save networks in **csv**, either as adjacency matrices or interaction lists, but these can loose some properties.

Exporting the adjacency matrix and the interaction list of a directed network:

In [ ]:
filename<-"./Output/FW_adjacency.csv"
A <- as.matrix(as_adjacency_matrix(FW, attr = "weight")) #one goes from graph to adjacency matrix
write.csv(A, filename) #and exports the adjacency matrix

In [ ]:
filename<-"./Output/FW_ilist.csv"
FW_Ilist <- igraph::as_data_frame(FW)
write.csv(FW_Ilist, filename,row.names = FALSE) #and exports the adjacency matrix

Exporting the incidence matrix a bipartite network:

In [ ]:
filename<-"./Output/B_incidence.csv"
I <- as.matrix(as_biadjacency_matrix(B))
write.csv(t(I), filename) #exports the incidence matrix (use transpose to have plants as rows!)

and the interaction list:

In [ ]:
filename <- "./Output/B_ilist.csv"
Ilist <- igraph::as_data_frame(B2)
colnames(Ilist)[1:2] <- c("plant_sp", "pol_sp")
write.csv(Ilist, filename,row.names = FALSE) #exports the incidence matrix (use transpose to have plants as rows!)

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 4 & 5 </h3>
Creathe the graph of the foodweb of the St. Marks stuary, and the mutualistic network from Rio Blaco from the provided files

</div>

In [ ]:
# Foodweb: continue your code here
FW_filename<- "./Data/FW_st_marks.csv"

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex4.R")



In [ ]:
# Pollination: continue your code here
P_Filename<-"./Data/Medan_Rio_Blanco.csv"

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex5.R")

-----------------------------------------------------

# Lesson 1B 

Now let's continue with basic functions of igraph to access graph elements, and basic properties:
- Obtaining nodes and edges
- Checking if node/edge exists
- Counting nodes and edges
- Degree of nodes

Now that we saw how one can create or load netwrosk in R, let's start to explore them. 

## Obtaining nodes and links in a network

We can get the number of nodes and edges in a graph using the `vcount(g)` and `ecount(g)` functions. 

We can get all the nodes(vertices) in a network using `V(g)` and all the links(edges) using `E(g)`.


In [ ]:
# Example: obtain all the nodes in the Mondego foodweb
FW_Afilename<-"./Data/Mondego_adjacency.csv"
df <- read.csv(FW_Afilename, row.names = 1)
A <- as.matrix(df)
FW <- graph_from_adjacency_matrix(
  A,
  mode = "directed",   # or "undirected"
  weighted = TRUE,     # FALSE if binary
)
FW

In [ ]:
rowSums(A)
colSums(A)

In [ ]:
# let's obtain the number of nodes and links
N=vcount(FW)
N
L=ecount(FW)
L

In [ ]:
#lest obtain all the nodes
V(FW)

you can also access the attributes with "$" (if they exists)

In [ ]:
V(FW)$name # for example the node names are an attribute

you can access **one particular node** either by its name, by its internal index, or by a condition 

In [ ]:
#by name
V(FW)["Phytoplankton"]

#by internal index
V(FW)[1]

#by a condition
V(FW)[name %in% c("Phytoplankton", "Zooplankton")]

We can also obtain the edges of a network in a similar way

In [ ]:
E(FW) #they are directed, notice the arrow pointing the direction of the interaction

And the attributes of the links as well

In [ ]:
E(FW)$weight 

One can see all the attributes that are present in the graph summary:

In [ ]:
FW

## Global and Local functions

Some functions take the full graph as variable, like `E(G)`, and return one value for the entire graph. 

However, other functions apply only to a node or a link. These are local functions and take more variables. 

### Neighbours
For example the function that returns the neighbours of a node `neighbors(g,v)`

In [ ]:
my_node=V(FW)[1] #the third node
neighbors(FW, my_node, mode = "all")


> Note: Because this is a **directed** network we have in-neighbours, and out-neighbours!

In [ ]:
neighbors(FW, my_node, mode = "out")
neighbors(FW, my_node, mode = "in")

### Node degree

One of the most important questions we can ask about a node in a graph is how many other nodes it connects to. Using the `neighbors()` function from above, we could formulate this question as so: how many neighbours does my node have?

In [ ]:
length(neighbors(FW, my_node, mode = "all"))

but this is such a common task that *igraph** provides us a graph method to do this in a much clearer way:

In [ ]:
igraph::degree(FW, my_node)

> In **directed networks** we have `in-degree()` (edges **entering** the node) and `out degree()` (edges **exiting** the node). The function `degree()` in directed networks returns the sum of the in and out connections.

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>
<h3> Exercise 6</h3>
Load the foodweb of the St Marks Estuary and answer these questions:
    
- How many species are in the network?
- What is the species that has more predators? 
- What is the species that has a more varied diet? 
- What are the species that feed on the most generalist predator? 
    
![title](./images/figure5.png)
</div>


In [ ]:
#your code here (remember you already loaded the network before, look how it is done)

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex6.R")

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>
<h3> Exercise 6B</h3>
Load the pollination network of Rio Blanco, and answer these questions:

- How many plant and pollinator species are in the network?
- What is the most generalist polinator?
- How many specialized plants arethere? (k=1)
- How many pollinators are shared between the two most generalist plants?

![title](./images/figure6.png)

In [ ]:
#your code here (remember you already loaded the network before, look how it is done)

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex6B.R")